# 9.1 モデルサービングAPI クライアント

`9.1.ipynb` で起動した MLflow モデルサービング API にリクエストを送り、文書情報抽出モデルのレスポンスを確認するための notebook です。

## 前提

- `9.1.ipynb` のサービングセルが実行中であること
- モデルAPIが `http://localhost:5001` で起動していること
- サーバー側の環境に `OPENAI_API_KEY` が設定されていること

In [ ]:
# ---- このセルの見どころ ----
# サービングAPIのエンドポイントを定義します。
# MLflow PyFunc Servingでは、推論リクエストは /invocations にPOSTします。
# 9.1.ipynbではTracking Serverを5000、モデルAPIを5001に分けています。

from __future__ import annotations

import json
from typing import Any

import pandas as pd
import requests
from IPython.display import display

BASE_URL = "http://localhost:5001"
INVOCATIONS_URL = f"{BASE_URL}/invocations"
DEFAULT_MODEL = "gpt-5-nano-2025-08-07"

## 1. サーバーの疎通確認

`/ping` または `/health` が `200` を返せば、サービングプロセスは起動しています。ここでは推論までは行いません。

In [ ]:
# ---- このセルの見どころ ----
# まずAPIサーバーが起動しているかだけを確認します。
# ここで接続エラーになる場合は、9.1.ipynbのサービングセルを実行してください。
# サーバーが起動済みなら、次のセルで実際の推論リクエストを送ります。

def check_server(base_url: str = BASE_URL) -> None:
    for path in ("/ping", "/health"):
        url = f"{base_url}{path}"
        try:
            response = requests.get(url, timeout=5)
        except requests.RequestException as exc:
            print(f"{url}: 接続できませんでした ({exc})")
            continue

        print(f"{url}: status={response.status_code}")
        if response.ok:
            print("サービングAPIは起動しています。")
            return

    raise RuntimeError("サービングAPIに接続できません。9.1.ipynbのサービングセルを確認してください。")


check_server()

## 2. 1件だけ推論する

MLflow PyFunc Servingには `dataframe_split` 形式で入力します。このモデルは `text` と `model` カラムを受け取ります。

In [ ]:
# ---- このセルの見どころ ----
# requests.postで /invocations に入力データを送ります。
# サーバー側でOpenAI APIが呼ばれ、抽出結果がJSONとして返ります。
# response.raise_for_status() により、4xx/5xxの失敗時は本文付きで原因を確認できます。

def invoke_document_extraction(rows: list[dict[str, str]], timeout: int = 120) -> dict[str, Any]:
    payload = {
        "dataframe_split": {
            "columns": ["text", "model"],
            "data": [[row["text"], row.get("model", DEFAULT_MODEL)] for row in rows],
        }
    }

    response = requests.post(INVOCATIONS_URL, json=payload, timeout=timeout)
    if not response.ok:
        print("Request failed")
        print("status:", response.status_code)
        print(response.text)
        response.raise_for_status()

    return response.json()


sample_text = "契約者は株式会社サンプルで、契約期間は2025年1月1日から2025年12月31日までです。サービスプランはプレミアムで、月額120,000円です。"

result = invoke_document_extraction([
    {"text": sample_text, "model": DEFAULT_MODEL},
])

print(json.dumps(result, ensure_ascii=False, indent=2))

## 3. 結果を表で見る

MLflow Servingのレスポンスは通常 `predictions` キーに入ります。表形式に変換すると、抽出された項目を確認しやすくなります。

In [ ]:
# ---- このセルの見どころ ----
# 返却形式に合わせて、predictionsをDataFrameに変換します。
# もしレスポンス形式が変わっても、上のセルの生JSONを見れば切り分けできます。

predictions = result.get("predictions", result)
display(pd.DataFrame(predictions))

## 4. 複数件をまとめて送る

同じ `dataframe_split` 形式で複数行を送ると、まとめて推論できます。

In [ ]:
# ---- このセルの見どころ ----
# バッチ推論の例です。
# DataFrameの各行がモデルのpredict内で1件ずつ処理されます。
# 本番では大量データを一度に送りすぎず、タイムアウトやAPI制限に合わせて分割します。

batch_rows = [
    {
        "text": "契約者は株式会社アルファです。契約開始日は2026年4月1日、終了日は2027年3月31日です。スタンダードプランを月額80,000円で利用します。",
        "model": DEFAULT_MODEL,
    },
    {
        "text": "株式会社ベータは、エンタープライズプランを2026年5月15日から2026年11月14日まで契約します。月額料金は250,000円です。",
        "model": DEFAULT_MODEL,
    },
]

batch_result = invoke_document_extraction(batch_rows)
display(pd.DataFrame(batch_result.get("predictions", batch_result)))

## トラブルシュート

- `Connection refused`: `9.1.ipynb` のサービングセルが起動しているか確認してください。
- `404`: URLが `http://localhost:5001/invocations` になっているか確認してください。
- `401` または OpenAI API エラー: サービングを起動した環境の `OPENAI_API_KEY` を確認してください。
- レスポンスが遅い: サーバー側でOpenAI APIを呼んでいるため、初回や混雑時は時間がかかることがあります。